# PIB — Step 1: Tesseract OCR (word text + boxes + text position)

Runs [Tesseract](https://github.com/tesseract-ocr/tesseract) (CPU) on every page of
[`disi-unibo-nlp/paint-it-black`](https://huggingface.co/datasets/disi-unibo-nlp/paint-it-black)
and saves, for each word:

- `text` — the word as read by Tesseract
- `bbox` — `[y_min, x_min, y_max, x_max]` in `[0, 1]` (same convention as the PIB ground truth)
- `start`, `end` — character offsets in the page text, so `page["text"][start:end] == word["text"]`

Output: one JSONL file per split (`base.jsonl`, `medium.jsonl`, `hard.jsonl`), one line per page, in dataset order.

```json
{"split": "base", "idx": 0, "source_pdf": "...", "page": 1, "total_pages": 1, "doc_type": "mri_reports",
 "width": 1190, "height": 1682,
 "text": "PATIENT NAME: MARCUS J STERLING\nDOB: ...",
 "words": [{"i": 0, "text": "PATIENT", "bbox": [0.1349, 0.0714, 0.1456, 0.1479],
            "start": 0, "end": 7, "conf": 96.0, "block": 2, "par": 1, "line": 1}, ...]}
```

`block / par / line` are Tesseract's layout ids: words that share all three are on the same text line
(useful later to build one box per line for multi-line entities).
No GPU is needed for this step.

In [ ]:
# 1. Install
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install pytesseract datasets
!tesseract --version | head -1

In [ ]:
# 2. Settings
DATASET     = "disi-unibo-nlp/paint-it-black"
SPLITS      = ["base", "medium", "hard"]
OUT_DIR     = "/content/pib_ocr/tesseract"
LANG        = "eng"
TESS_CONFIG = "--oem 1 --psm 3"   # LSTM engine, automatic page segmentation
UPSCALE     = 1.0                 # 1.0 = OCR the image as stored (144 DPI). Boxes are normalised, so any value is safe.
N_WORKERS   = 2                   # Colab has 2 CPU cores
MAX_PAGES   = None                # e.g. 5 for a quick test; None = all pages
SAVE_TO_DRIVE = False             # True = also copy the results to Google Drive at the end

In [ ]:
# 3. Load the dataset
from datasets import load_dataset

try:
    dsd = load_dataset(DATASET)
except Exception as exc:
    # Private or gated repo: add your token as a Colab secret named HF_TOKEN (key icon on the left) and re-run.
    print(f"Anonymous load failed ({type(exc).__name__}); retrying with HF_TOKEN from Colab secrets...")
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    dsd = load_dataset(DATASET)

for s in SPLITS:
    print(f"{s:7s} {len(dsd[s]):4d} pages   columns: {dsd[s].column_names}")

In [ ]:
# 4. OCR functions
import os
os.environ["OMP_THREAD_LIMIT"] = "1"   # one thread per Tesseract process; we parallelise over pages instead

import json, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pytesseract
from pytesseract import Output
from PIL import Image


def ocr_page(img, lang=LANG, config=TESS_CONFIG, upscale=UPSCALE):
    """Run Tesseract on one PIL image.

    Returns (text, words, (width, height)). Each word has its text, a normalised
    [y_min, x_min, y_max, x_max] box, and [start, end) character offsets into `text`.
    """
    img = img.convert("RGB")
    orig_size = img.size
    if upscale != 1.0:
        img = img.resize((round(img.width * upscale), round(img.height * upscale)), Image.LANCZOS)
    W, H = img.size

    # Output.DICT keeps every token as a string (a DataFrame would turn words like "NA" into NaN).
    d = pytesseract.image_to_data(img, lang=lang, config=config, output_type=Output.DICT)

    words, parts, pos, prev = [], [], 0, None
    for k in range(len(d["text"])):
        if d["level"][k] != 5:                 # 5 = word level
            continue
        tok = (d["text"][k] or "").strip()
        if not tok:
            continue
        key = (d["block_num"][k], d["par_num"][k], d["line_num"][k])
        if prev is None:
            sep = ""
        elif key == prev:
            sep = " "                          # same line
        elif key[0] == prev[0]:
            sep = "\n"                         # new line, same block
        else:
            sep = "\n\n"                       # new block
        parts.append(sep)
        pos += len(sep)

        left, top, w, h = d["left"][k], d["top"][k], d["width"][k], d["height"][k]
        clip = lambda v: round(min(1.0, max(0.0, v)), 5)
        words.append({
            "i":     len(words),
            "text":  tok,
            "bbox":  [clip(top / H), clip(left / W), clip((top + h) / H), clip((left + w) / W)],
            "start": pos,
            "end":   pos + len(tok),
            "conf":  float(d["conf"][k]),
            "block": key[0], "par": key[1], "line": key[2],
        })
        parts.append(tok)
        pos += len(tok)
        prev = key

    return "".join(parts), words, orig_size


def _ocr_row(split, idx, row):
    text, words, (width, height) = ocr_page(row["image"])
    return {
        "split": split, "idx": idx,
        "source_pdf": row.get("source_pdf"), "page": row.get("page"),
        "total_pages": row.get("total_pages"), "doc_type": row.get("doc_type"),
        "width": width, "height": height,
        "text": text, "words": words,
    }


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def ocr_split(ds, split, out_dir=OUT_DIR, max_pages=MAX_PAGES, n_workers=N_WORKERS):
    """OCR one split and append one JSON line per page. Safe to re-run: finished pages are skipped."""
    out_path = Path(out_dir) / f"{split}.jsonl"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    n = len(ds) if max_pages is None else min(max_pages, len(ds))
    n_done = len(read_jsonl(out_path)) if out_path.exists() else 0   # pages are written in order
    if n_done >= n:
        print(f"[{split}] already complete ({n_done} pages) -> {out_path}")
        return out_path

    t0 = time.time()
    with ThreadPoolExecutor(max_workers=n_workers) as pool, open(out_path, "a", encoding="utf-8") as f:
        for start in range(n_done, n, 8):                              # small batches keep memory low
            idxs = list(range(start, min(start + 8, n)))
            for rec in pool.map(lambda i: _ocr_row(split, i, ds[i]), idxs):
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            f.flush()
            done = idxs[-1] + 1
            print(f"\r[{split}] {done}/{n} pages  ({(time.time() - t0) / (done - n_done):.1f} s/page)", end="")
    print(f"\n[{split}] saved -> {out_path}")
    return out_path

In [ ]:
# 5. Run OCR on every split (roughly 10-20 minutes for all 414 pages on Colab CPU)
out_paths = {s: ocr_split(dsd[s], s) for s in SPLITS}

meta = {
    "dataset": DATASET, "splits": SPLITS, "engine": "tesseract",
    "tesseract_version": str(pytesseract.get_tesseract_version()),
    "lang": LANG, "config": TESS_CONFIG, "upscale": UPSCALE,
    "bbox_format": "[y_min, x_min, y_max, x_max], normalised to [0, 1] by page height (y) and width (x)",
    "offsets": "page['text'][word['start']:word['end']] == word['text']",
    "text_separators": {"same line": " ", "new line": "\\n", "new block": "\\n\\n"},
}
Path(OUT_DIR, "meta.json").write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))

In [ ]:
# 6. Integrity checks on the saved files + summary
ocr = {s: read_jsonl(out_paths[s]) for s in SPLITS}

for s in SPLITS:
    pages = ocr[s]
    n_expected = len(dsd[s]) if MAX_PAGES is None else min(MAX_PAGES, len(dsd[s]))
    assert len(pages) == n_expected, f"{s}: {len(pages)} pages saved, expected {n_expected}"
    assert [p["idx"] for p in pages] == list(range(len(pages))), f"{s}: pages are not in dataset order"
    for p in pages:
        for w in p["words"]:
            assert p["text"][w["start"]:w["end"]] == w["text"], (s, p["idx"], w)
            y0, x0, y1, x1 = w["bbox"]
            assert 0 <= y0 <= y1 <= 1 and 0 <= x0 <= x1 <= 1, (s, p["idx"], w)
    n_words = sum(len(p["words"]) for p in pages)
    confs = [w["conf"] for p in pages for w in p["words"]]
    empty = sum(1 for p in pages if not p["words"])
    print(f"{s:7s} pages={len(pages):4d}  words={n_words:6d}  words/page={n_words / len(pages):6.1f}  "
          f"mean conf={sum(confs) / max(1, len(confs)):5.1f}  conf<50: {sum(c < 50 for c in confs):5d}  "
          f"pages with no words: {empty}")
print("Offsets and box ranges are consistent on every page.")

In [ ]:
# 7. Visual check: draw the word boxes on 3 pages
import matplotlib.pyplot as plt
from PIL import ImageDraw

CHECK = [("base", 0), ("medium", 0), ("hard", 0)]   # (split, page index) - change freely


def draw_words(img, words, min_conf=None):
    img = img.convert("RGB").copy()
    W, H = img.size
    draw = ImageDraw.Draw(img)
    for w in words:
        y0, x0, y1, x1 = w["bbox"]
        low = min_conf is not None and w["conf"] < min_conf
        draw.rectangle([x0 * W, y0 * H, x1 * W, y1 * H], outline=(230, 120, 0) if low else (220, 0, 0), width=1)
    return img


check_dir = Path(OUT_DIR) / "check"
check_dir.mkdir(parents=True, exist_ok=True)
for split, idx in CHECK:
    page = ocr[split][idx]
    vis = draw_words(dsd[split][idx]["image"], page["words"], min_conf=50)
    vis.save(check_dir / f"{split}_{idx:04d}.png")
    plt.figure(figsize=(12, 17))
    plt.imshow(vis)
    plt.axis("off")
    plt.title(f"{split} / page {idx} / {page['source_pdf']}  -  {len(page['words'])} words "
              f"(red = word box, orange = confidence < 50)")
    plt.show()
    print(page["text"][:600], "\n" + "-" * 80)

In [ ]:
# 8. Check against the ground truth (all pages): do the OCR words fall inside the annotated PHI boxes?
#    For each GT entity we collect the OCR words whose centre lies inside one of its boxes and compare
#    their joined text with the GT text. This also shows the best an OCR-based pipeline can reach.
import string


def _as_records(anns):
    """Annotations as a list of dicts (handles both list-of-dicts and dict-of-lists layouts)."""
    if isinstance(anns, dict):
        keys = list(anns)
        return [dict(zip(keys, vals)) for vals in zip(*(anns[k] for k in keys))]
    return anns or []


def words_in_boxes(words, bboxes, tol=0.002):
    hits = []
    for w in words:
        cy, cx = (w["bbox"][0] + w["bbox"][2]) / 2, (w["bbox"][1] + w["bbox"][3]) / 2
        if any(b[0] - tol <= cy <= b[2] + tol and b[1] - tol <= cx <= b[3] + tol for b in bboxes):
            hits.append(w)
    return hits


_norm = lambda t: " ".join(t.lower().split())
_trim = lambda t: " ".join(tok.strip(string.punctuation) for tok in _norm(t).split()).strip()

print(f"{'split':7s} {'GT entities':>11s} {'located':>9s} {'exact text':>11s} {'exact (edge punct. ignored)':>29s}")
for s in SPLITS:
    gt_rows = dsd[s].select_columns(["annotations"])       # skips image decoding
    n = located = exact = loose = 0
    for page in ocr[s]:
        for ann in _as_records(gt_rows[page["idx"]]["annotations"]):
            hits = words_in_boxes(page["words"], ann["bboxes"])
            got = " ".join(w["text"] for w in hits)
            n += 1
            located += bool(hits)
            exact += _norm(got) == _norm(ann["text"])
            loose += _trim(got) == _trim(ann["text"])
    print(f"{s:7s} {n:11d} {located / n:9.1%} {exact / n:11.1%} {loose / n:29.1%}")

In [ ]:
# 9. Package the results
import shutil

zip_path = shutil.make_archive("/content/pib_ocr_tesseract", "zip", OUT_DIR)
print("Zipped:", zip_path)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/pib_ocr/tesseract"
    shutil.copytree(OUT_DIR, dest, dirs_exist_ok=True)
    print("Copied to:", dest)
else:
    from google.colab import files
    files.download(zip_path)

## Reading the file in the next steps

```python
import json
pages = [json.loads(l) for l in open("/content/pib_ocr/tesseract/base.jsonl", encoding="utf-8")]
page = pages[0]
page["text"]                      # full page text (input for GLiNER / LLM taggers)
w = page["words"][0]
page["text"][w["start"]:w["end"]] # == w["text"]
w["bbox"]                         # [y_min, x_min, y_max, x_max] in [0, 1]
```

To turn a predicted character span `[a, b)` into boxes: take the words with `start < b and end > a`,
group them by `(block, par, line)`, and take the enclosing rectangle of each group (one box per line).